# Research data supporting "Is fixed node diffusion quantum Monte Carlo reproducible?"

This notebook accompanies the paper: **Is fixed node diffusion quantum Monte Carlo reproducible?**. It can be found on Github at: https://github.com/zenandrea/DMC-codes-comparison and also on [Colab](https://colab.research.google.com/github/zenandrea/DMC-codes-comparison/blob/main/analysis.ipynb)

### Abstract

Fixed-node quantum diffusion Monte Carlo (FN-DMC) is a widely-trusted many-body method for solving the Schr\"{o}dinger equation, known for its reliable predictions of material and molecular properties. Furthermore, its excellent scalability with system complexity and near-perfect utilization of computational power makes FN-DMC ideally positioned to leverage new advancements in computing to address increasingly complex scientific problems. However, demonstrating reproducibility across different FN-DMC code implementations remains an outstanding challenge. This difficulty stems from the diverse array of DMC algorithms and wave-function implementations, compounded by the method's inherent stochastic nature. This study represents a community-wide effort to address the titular question, affirming that: Yes, FN-DMC is reproducible. Using the water-methane dimer as the canonical test case, we identified the treatment of the non-locality of pseudopotentials as the primary source of discrepancies between different implementations. In particular, we demonstrate that while the locality approximation (LA) leads to considerable variability in results, reliable and reproducible predictions can be achieved by employing the T-move (TM), the determinant locality approximation (DLA), or the determinant locality T-move (DTM) schemes. This work lays the foundation to establish accurate and reproducible FN-DMC estimates for all future studies across its countless applications in materials science, physics, chemistry, and biology.

In [1]:
# Check if we are in Google Colab environment
try:
    import google.colab
    IN_COLAB = True
    usetex = False
except:
    import os
    IN_COLAB = False
    if os.path.expanduser('~') == '/home/shixubenjamin':
        usetex = True
    else:
        usetex = False

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy.stats as stats
from scipy.optimize import curve_fit

from Scripts.myfit import *
from Scripts.myfunctions import *
from Scripts.mysetup import *
from Scripts.jup_plot import *

# Hartree to meV
Ha2meV = 27.211396641308*1000

if usetex:
    textrue_import()
else:
    texfalse_import()

## CCSDT(Q) reference

In [3]:
cc_bind_ene_cp = {x:0 for x in ['DZ','TZ','QZ','5Z','CBS(DZ/TZ)','CBS(TZ/QZ)','CBS(QZ/5Z)']}
cc_bind_ene_nocp = {x:0 for x in ['DZ','TZ','QZ','5Z','CBS(DZ/TZ)','CBS(TZ/QZ)','CBS(QZ/5Z)']}

ene_hf_cp = []
ene_ccsdt_cp = []
ene_hf_nocp = []
ene_ccsdt_nocp = []

# Read the calculation outputs from MRCC. We calculate using counterpoise correction (cp) and without counterpoise correction (nocp) at the CBS(QZ/5Z) for CCSD(T) and DZ level for CCSDT(Q). Final results are the average of cp and nocp.
for i in ['DZ','TZ','QZ','5Z']:
    ene_hf_cp +=  [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_cp/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_cp/mrcc.out')['scf_energy'])*1000]
    ene_hf_nocp += [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_nocp/mrcc.out')['scf_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_nocp/mrcc.out')['scf_energy'])*1000]
    ene_ccsdt_cp += [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_cp/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_cp/mrcc.out')['ccsd(t)_corr_energy'])*1000]
    ene_ccsdt_nocp += [(read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_nocp/mrcc.out')['ccsd(t)_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_nocp/mrcc.out')['ccsd(t)_corr_energy'])*1000]

    cc_bind_ene_cp[i] = (read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_cp/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_cp/mrcc.out')['energy'])*1000
    cc_bind_ene_nocp[i] = (read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/mw/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/m_nocp/mrcc.out')['energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDT/{i}/w_nocp/mrcc.out')['energy'])*1000

# Let's get CCSD(T) CBS(DZ/TZ)
cc_bind_ene_cp['CBS(DZ/TZ)'] = get_cbs(ene_hf_cp[0],ene_ccsdt_cp[0],ene_hf_cp[1],ene_ccsdt_cp[1],X=2,Y=3,output=False)[-1]
cc_bind_ene_nocp['CBS(DZ/TZ)'] = get_cbs(ene_hf_nocp[0],ene_ccsdt_nocp[0],ene_hf_nocp[1],ene_ccsdt_nocp[1],X=2,Y=3,output=False)[-1]

# Let's get CCSD(T) CBS(TZ/QZ)
cc_bind_ene_cp['CBS(TZ/QZ)'] = get_cbs(ene_hf_cp[1],ene_ccsdt_cp[1],ene_hf_cp[2],ene_ccsdt_cp[2],X=3,Y=4,output=False)[-1]
cc_bind_ene_nocp['CBS(TZ/QZ)'] = get_cbs(ene_hf_nocp[1],ene_ccsdt_nocp[1],ene_hf_nocp[2],ene_ccsdt_nocp[2],X=3,Y=4,output=False)[-1]

# Let's get CCSD(T) CBS(QZ/5Z)
cc_bind_ene_cp['CBS(QZ/5Z)'] = get_cbs(ene_hf_cp[2],ene_ccsdt_cp[2],ene_hf_cp[3],ene_ccsdt_cp[3],X=4,Y=5,output=False)[-1]
cc_bind_ene_nocp['CBS(QZ/5Z)'] = get_cbs(ene_hf_nocp[2],ene_ccsdt_nocp[2],ene_hf_nocp[3],ene_ccsdt_nocp[3],X=4,Y=5,output=False)[-1]

# Let's get CCSDT(Q) DZ
ene_ccsdtq_cp = 0
ene_ccsdtq_nocp = 0

ene_ccsdtq_cp = (read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/mw/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/m_cp/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/w_cp/mrcc.out')['ccsdtq_corr_energy'])*1000
ene_ccsdtq_nocp = (read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/mw/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/m_nocp/mrcc.out')['ccsdtq_corr_energy'] - read_mrcc_output(f'Codes_Data/MRCC/CCSDTQ/DZ/w_nocp/mrcc.out')['ccsdtq_corr_energy'])*1000

final_ccsdtq_cp = ene_ccsdtq_cp - ene_ccsdt_cp[0] + cc_bind_ene_cp['CBS(QZ/5Z)']
final_ccsdtq_nocp = ene_ccsdtq_nocp - ene_ccsdt_nocp[0] + cc_bind_ene_nocp['CBS(QZ/5Z)']
final_ccsdtq_ref = (final_ccsdtq_cp + final_ccsdtq_nocp) / 2
final_ccsdt_ref = (cc_bind_ene_cp['CBS(QZ/5Z)'] + cc_bind_ene_nocp['CBS(QZ/5Z)']) / 2
print(f'Final CCSDT(Q) binding energy: {final_ccsdtq_ref:.2f} meV')

Final CCSDT(Q) binding energy: -26.61 meV


## Analysis of DMC code data

In [14]:
# Defining variables and colours


systems = [ 'm', 'w', 'mw', 'm---w' ]
methods = [ 'LA', 'TM', 'DLA', 'DLTM' ]

# Define dictionary with the methods implemented within each code (alongside folder location) and the corresponding label and color
dmc_code_info ={
    'Amolqc': {'methods_and_folders': {x: 'Amolqc' for x in ['DLA']}, 'label': 'Amolqc', 'color': '#e6194B', 'discretization': 'tau'},
    'CASINO': {'methods_and_folders': {x: 'CASINO_limdmc5' for x in['LA','TM','DLA','DLTM']}, 'label': 'CASINO', 'color': '#f58231', 'discretization': 'tau'},
    'CHAMP_Cornell_Paris': {'methods_and_folders': {x:'CHAMP_Cornell_Paris' for x in ['LA','TM']}, 'label': 'CHAMP (Cornell-Paris)', 'color': '#ffe119', 'discretization': 'tau'},
    'CHAMP_Holland': {'methods_and_folders': {x:'CHAMP_Holland' for x in ['LA','TM']} , 'label': 'CHAMP (Holland)', 'color': '#bfef45', 'discretization': 'tau'},
    'CMQMC': {'methods_and_folders': {x:'CMQMC' for x in ['LA','TM','DLA','DLTM']}, 'label': 'CMQMC', 'color': '#3cb44b', 'discretization': 'tau'},
    'PyQMC': {'methods_and_folders': {x:'PyQMC' for x in ['LA','TM','DLA']}, 'label': 'PyQMC', 'color': '#42d4f4', 'discretization': 'tau'},
    'QMC=Chem': {'methods_and_folders': {x:'QMC=Chem' for x in ['DLA']}, 'label': 'QMC=Chem', 'color': '#4363d8', 'discretization': 'tau'},
    'QMCPACK': {'methods_and_folders': {'LA': 'QMCPACK_complex' , 'TM': 'QMCPACK', 'DLA': 'QMCPACK'}, 'label': 'QMCPACK', 'color': '#911eb4', 'discretization': 'tau'},
    'QMeCha': {'methods_and_folders': {x:'QMeCha' for x in ['LA','DLA']}, 'label': 'QMeCha', 'color': '#f032e6', 'discretization': 'tau'},
    'QWalk': {'methods_and_folders': {x:'QWalk_noncov' for x in ['LA','TM','DLA']}, 'label': 'QWalk', 'color': '#800000', 'discretization': 'tau'},
    'TurboRVB_DMC': {'methods_and_folders': {x:'TurboRVB_DMCv2' for x in ['TM']}, 'label': 'TurboRVB (DMC)', 'color': '#469990', 'discretization': 'tau'},
    'TurboRVB_LRDMC': {'methods_and_folders': {x:'TurboRVB' for x in ['TM','DLA','DLTM']}, 'label': 'TurboRVB (LRDMC)', 'color': '#808000', 'discretization': 'a'}
}

method_codes_dict = {
    'LA': [code for code in dmc_code_info.keys() if 'LA' in dmc_code_info[code]['methods_and_folders'].keys()],
    'TM': [code for code in dmc_code_info.keys() if 'TM' in dmc_code_info[code]['methods_and_folders'].keys()],
    'DLA': [code for code in dmc_code_info.keys() if 'DLA' in dmc_code_info[code]['methods_and_folders'].keys()],
    'DLTM': [code for code in dmc_code_info.keys() if 'DLTM' in dmc_code_info[code]['methods_and_folders'].keys()]
}

code_to_num_dict = {x: index for index, x in enumerate(dmc_code_info.keys())}

In [20]:
# Initialize dictionary to store the data. The dictionary is structured such that the keys are the codes and the values are dictionaries with keys 'raw', 'fit' and 'final'. The 'raw' key contains the raw data from the calculations containing the values at the calculated timesteps, 'fit' contains the fitted data and 'final' contains the final binding energy extrapolated to the zero time step limit.

dmc_code_data = {code: {quantity: {'raw': {}, 'fit': {}, 'final': {}} for quantity in ['total_energy','binding_energy', 'binding_energy_far','size_consistency_error']} for code in dmc_code_info.keys()}
final_binding_energy = {code: {} for code in dmc_code_info.keys()}

for code in dmc_code_info.keys():
    
    for method, method_folder in dmc_code_info[code]["methods_and_folders"].items():
        for system in systems:
            key = f'{system}_{method}'
            # Read in the raw total energy data from the csv files
            raw_data = pd.read_csv(
                f'Codes_Data/{method_folder}/energy_{key}.csv',
                names=[dmc_code_info[code]["discretization"], 'ene','err'], decimal='.', header=None, index_col=dmc_code_info[code]["discretization"])
            dmc_code_data[code]['total_energy']['raw'][key] = raw_data


        # Calculate the binding energy, binding energy (far) and size consistency error from the total energy of the methane-water dimer, methane, water and the separate methane-water system.

        mw = dmc_code_data[code]['total_energy']['raw'][f'mw_{method}']
        m = dmc_code_data[code]['total_energy']['raw'][f'm_{method}']
        w = dmc_code_data[code]['total_energy']['raw'][f'w_{method}']
        mw_far = dmc_code_data[code]['total_energy']['raw'][f'm---w_{method}']

        dmc_code_data[code]['binding_energy']['raw'][method] = pd.DataFrame([Ha2meV*(mw.ene-m.ene-w.ene), Ha2meV*(mw.err**2+m.err**2+w.err**2)**0.5]).T.dropna()
        dmc_code_data[code]['binding_energy_far']['raw'][method] = pd.DataFrame([Ha2meV*(mw.ene - mw_far.ene), Ha2meV*(mw.err**2+mw_far.err**2)**0.5]).T.dropna()
        dmc_code_data[code]['size_consistency_error']['raw'][method] = pd.DataFrame([Ha2meV*(mw_far.ene-m.ene-w.ene), Ha2meV*(mw_far.err**2+m.err**2+w.err**2)**0.5]).T.dropna()

        # Fit the the binding energy and total energy quantities
        for energy_quantity in ['mw','mw_far','m','w','binding_energy','binding_energy_far','size_consistency_error']:
            if energy_quantity in ['mw','m','w']:
                fitting_data = dmc_code_data[code]['total_energy']['raw'][f'{energy_quantity}_{method}']
            elif energy_quantity in ['binding_energy','binding_energy_far','size_consistency_error']:
                fitting_data = dmc_code_data[code][energy_quantity]['raw'][method]
            elif energy_quantity == 'mw_far':
                fitting_data = dmc_code_data[code]['total_energy']['raw'][f'm---w_{method}']

            fitting_data['x'] = x_axis( fitting_data, code )
            fitting_data_trunc = fitting_data[ fitting_data.x <= taumax ].dropna()

            if code in ['TurboRVB_DMC','QMC=Chem']:
                fitting_function = fun_cub
            elif code == 'CHAMP_Holland':
                fitting_function = fun_4
            else:
                fitting_function = fun_quad

            xdata = fitting_data_trunc.x.to_numpy()
            ydata = fitting_data_trunc.ene.to_numpy()
            sigma = fitting_data_trunc.err.to_numpy()
            popt, pcov = curve_fit( fitting_function, xdata=xdata, ydata=ydata, sigma=sigma )

            if energy_quantity == 'binding_energy':
                chi2, ndof, alpha = get_chi2_alpha_fitfun( xdata, ydata, sigma, fitting_function, popt )
                xdata_tausmall = np.array([xdata[index] for index in range(len(xdata)) if xdata[index] < 0.011])
                ydata_tausmall = np.array([ydata[index] for index in range(len(xdata)) if xdata[index] < 0.011])
                sigma_tausmall = np.array([sigma[index] for index in range(len(xdata)) if xdata[index] < 0.011])
                wmad = get_weighted_mad(xdata_tausmall, ydata_tausmall, sigma_tausmall, fitting_function, popt )

                y_extrap_zero = fitting_function(0, *popt)
                x_smallest_tau = np.min(xdata)
                y_smallest_tau = ydata[np.argmin(xdata)]
                print(f'{code.ljust(20)} {method.ljust(4)} ({round(x_smallest_tau,4):5.4f}) {chi2/ndof:-5.2f} {wmad:-5.2f} {y_extrap_zero:-5.2f} {y_smallest_tau:-5.2f}' )


            space_pts = 101
            x_linspace = np.linspace( 0, taumax, space_pts)
            energy_linspace = fitting_function( x_linspace, *popt )

            # Fit error curve to the data
            xfit, m, s = fit_err(xdata,ydata,sigma,fitfun=fitting_function,xmax=taumax,Nx=space_pts)

            # Getting the quantities in the limit of timestep = 0
            lim_Eb = energy_linspace[0]
            lim_Eb_upper = (m+1*s)[0]
            lim_Eb_lower = (m-1*s)[0]
            lim_Eb_sigma = np.max([abs(lim_Eb - lim_Eb_upper), abs(lim_Eb - lim_Eb_lower)])

            if energy_quantity in ['mw','m','w']:
                dmc_code_data[code]['total_energy']['fit'][f'{energy_quantity}_{method}'] = {'x': xdata, 'E': ydata , 'E_err': sigma, 'x_fit': xfit, 'E_fit': energy_linspace, 'E_fit_lower_lim': m-1*s, 'E_fit_upper_lim': m+1*s}
                dmc_code_data[code]['total_energy']['final'][f'{energy_quantity}_{method}'] = [lim_Eb, lim_Eb_sigma]
            elif energy_quantity in ['binding_energy','binding_energy_far','size_consistency_error']:
                dmc_code_data[code][energy_quantity]['fit'][f'{method}'] = {'x': xdata, 'E': ydata , 'E_err': sigma, 'x_fit': x_linspace, 'E_fit': energy_linspace ,  'E_fit_lower_lim': m-1*s,  'E_fit_upper_lim': m+1*s}
                dmc_code_data[code][energy_quantity]['final'][method] = [lim_Eb, lim_Eb_sigma]
                if energy_quantity == 'binding_energy':
                    final_binding_energy[code][method] = [lim_Eb, lim_Eb_sigma]
            elif energy_quantity == 'mw_far':
                dmc_code_data[code]['total_energy']['fit'][f'm---w_{method}'] = {'x': xdata, 'E': ydata , 'E_err': sigma, 'x_fit': xfit, 'E_fit': energy_linspace, 'E_fit_lower_lim': m-1*s, 'E_fit_upper_lim': m+1*s}
                dmc_code_data[code]['total_energy']['final'][f'm---w_{method}'] = [lim_Eb, lim_Eb_sigma]



Amolqc               DLA  (0.0025)  0.16  0.68 -27.89 -29.76


/home/shixubenjamin/anaconda3/lib/python3.7/site-packages/scipy/optimize/minpack.py:834: OptimizeWarning: Covariance of the parameters could not be estimated
  category=OptimizeWarning)


CASINO               LA   (0.0050)  1.15  0.67 -32.77 -33.33
CASINO               TM   (0.0050)  0.67  0.90 -27.22 -27.22
CASINO               DLA  (0.0025)  1.72  1.30 -31.21 -30.29
CASINO               DLTM (0.0025)  1.24  1.09 -26.85 -27.59
CHAMP_Cornell_Paris  LA   (0.0050)  2.74  1.41 -25.82 -23.35
CHAMP_Cornell_Paris  TM   (0.0050)  1.60  1.10 -28.99 -26.76
CHAMP_Holland        LA   (0.0100)  1.63  0.50 -35.13 -25.39
CHAMP_Holland        TM   (0.0025)  0.37  0.69 -26.75 -24.62
CMQMC                LA   (0.0025)  2.03  1.07 -25.74 -25.72
CMQMC                TM   (0.0025)  1.46  0.99 -28.19 -29.33
CMQMC                DLA  (0.0025)  2.69  1.24 -27.89 -29.20
CMQMC                DLTM (0.0025)  0.18  0.53 -27.80 -27.67
PyQMC                LA   (0.0025)  0.42  1.51 -38.79 -48.76
PyQMC                TM   (0.0025)  0.32  1.60 -25.85 -33.70
PyQMC                DLA  (0.0025)  1.16  1.78 -35.16 -38.02
QMC=Chem             DLA  (0.0025)  2.17  1.42 -27.41 -24.39
QMCPACK              LA 

## Figure 1 - Comparison among different codes

In [21]:
fig , ax = plt.subplots(4,1,figsize=(6.67,8),dpi=600,constrained_layout=True,sharex=True,sharey=True)

counter = 0
for method in ['LA','TM','DLA','DLTM']:
    ax[counter].axhline(y=final_ccsdtq_ref,color='black',ls='--')

    for code in method_codes_dict[method]:
        ax[counter].errorbar(code_to_num_dict[code],final_binding_energy[code][method][0],final_binding_energy[code][method][1],fmt='o',capsize=6,ms=10,elinewidth=1.5,markeredgewidth=1,mec='black',c=dmc_code_info[code]['color'])

    ax[counter].axhline(mean_codes(final_binding_energy,method_codes_dict[method],method)[0],c='grey',ls='dashed',lw=2)
    ax[counter].fill_between(np.arange(-0.5,13.5,0.1),mean_codes(final_binding_energy,method_codes_dict[method],method)[0] + mean_codes(final_binding_energy,method_codes_dict[method],method)[1],mean_codes(final_binding_energy,method_codes_dict[method],method)[0] - mean_codes(final_binding_energy,method_codes_dict[method],method)[1] ,color='grey',alpha=.3,edgecolor='none')
    ax[counter].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
    ax[counter].set_xticklabels([])
    ax[counter].set_xlim([-0.5,11.5])
    ax[counter].set_ylim([-55,-15])
    ax[counter].set_ylabel('Binding energy [meV]')

    counter += 1

ax[0].set_title('Locality Approximation (LA)')
ax[1].set_title('T-Move (TM)')
ax[2].set_title('Determinant Locality Approximation (DLA)')
ax[3].set_title('Determinant Locality Approximation with T-Move (DTM)',fontsize=11)

ax[3].set_xticklabels([dmc_code_info[x]['label'] for x in code_to_num_dict],rotation=90) 

plt.savefig('Figures/Fig_01.png')

## Figure 2 - Distribution of errors for different methods

In [24]:
fig , ax = plt.subplots(figsize=(3.37,4),dpi=600,constrained_layout=True)

x=np.arange(-55,-20,0.1)
heightstep = 0.6

ax.axvline(final_ccsdtq_ref,ls='dashed',lw=2,c='black')

# Plot distribution for LA
y_dist, mean, error = gauss(x,final_binding_energy,method_codes_dict['LA'],'LA',dist_info=True)
ax.errorbar(mean,heightstep*3 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='blue')
ax.plot(x,y_dist+heightstep*3,label='LA', lw=2,c='blue')
ax.fill_between(x,y_dist+heightstep*3,heightstep*3,color='blue',alpha=0.2)
ax.axhline(heightstep*3,ls='dashed',lw=0.5,c='blue')
t = ax.text(-52,heightstep*3 + 0.1,'LA',fontsize=15)
t.set_bbox(dict(facecolor='blue', alpha=0.5, edgecolor='blue'))
ax.text(-45,heightstep*3 + 0.17,r'$\mu_\textrm{LA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep*3 + 0.07,r'$\sigma_\textrm{LA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for TM
y_dist, mean, error = gauss(x,final_binding_energy,method_codes_dict['TM'],'TM',dist_info=True)
ax.errorbar(mean,heightstep*2 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='orange')
ax.plot(x,y_dist+heightstep*2,label='TM', lw=2,c='orange')
ax.fill_between(x,y_dist+heightstep*2,heightstep*2,color='orange',alpha=0.2)
ax.axhline(heightstep*2,ls='dashed',lw=0.5,c='orange')
t = ax.text(-52,heightstep * 2 + 0.1,'TM',fontsize=15)
t.set_bbox(dict(facecolor='orange', alpha=0.5, edgecolor='orange'))
ax.text(-45,heightstep*2 + 0.17,r'$\mu_\textrm{TM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep*2 + 0.07,r'$\sigma_\textrm{TM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for DLA
y_dist, mean, error = gauss(x,final_binding_energy,method_codes_dict['DLA'],'DLA',dist_info=True)
ax.errorbar(mean,heightstep - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='red')
ax.plot(x,y_dist+heightstep,label='DLA', lw=2,c='red')
ax.fill_between(x,y_dist+heightstep,heightstep,color='red',alpha=0.2)
ax.axhline(heightstep,ls='dashed',lw=0.5,c='red')
t = ax.text(-52,heightstep + 0.1,'DLA',fontsize=15)
t.set_bbox(dict(facecolor='red', alpha=0.5, edgecolor='red'))
ax.text(-45,heightstep + 0.17,r'$\mu_\textrm{DLA} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45,heightstep + 0.07,r'$\sigma_\textrm{DLA} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot distribution for DLTM
y_dist, mean, error = gauss(x,final_binding_energy,method_codes_dict['DLTM'],'DLTM',dist_info=True)
ax.errorbar(mean, -0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='green')
ax.plot(x,y_dist,label='DLTM', lw=2,c='green')
ax.fill_between(x,y_dist,0,color='green',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='green')
t = ax.text(-52,0.1,'DLTM',fontsize=15)
t.set_bbox(dict(facecolor='green', alpha=0.5, edgecolor='green'))
ax.text(-45, 0.17,r'$\mu_\textrm{DTM} \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-45, 0.07,r'$\sigma_\textrm{DTM} \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


ax.spines[['right', 'top','left']].set_visible(False)

plt.ylim([-0.2,2.05])
plt.yticks([])
plt.xlim([-55,-20])

plt.xlabel('Binding energy [meV]')
plt.savefig('Figures/Fig_02.png')


## Figure 3 - Timestep dependence of distribution of codes

In [29]:
fig , ax = plt.subplots(figsize=(3.37,5),dpi=600)

x=np.arange(-85,-10,0.1)
offset=0.5

ax.axvline(final_ccsdtq_ref,ls='dashed',lw=2,c='black')

# Codes which use the same set of timesteps for comparison
code_list_tau_DLA=['Amolqc','CASINO','CMQMC','PyQMC','QMC=Chem','QMCPACK']
DLA_binding_energy = {code: dmc_code_data[code]['binding_energy']['raw']['DLA'] for code in code_list_tau_DLA}


# Plot data for 0.0400 timestep
y_dist, mean, error = gauss_tau(x,DLA_binding_energy,code_list_tau_DLA,0.0400)
ax.errorbar(mean,offset*5 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='blue')
ax.plot(x,y_dist+offset*5,label=r'$\tau$ = 0.04 au',lw=2,c='blue')
ax.fill_between(x,y_dist+offset*5,offset*5,color='blue',alpha=0.2)
ax.axhline(offset*5,ls='dashed',lw=0.5,c='blue')
t = ax.text(-75,0.12+offset*5,r'$\tau$ = 0.04 au',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='blue'))
ax.text(-48,offset*5 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*5 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)

# Plot data for 0.0200 timestep
y_dist, mean, error = gauss_tau(x,DLA_binding_energy,code_list_tau_DLA,0.0200)
ax.errorbar(mean,offset*4 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='dodgerblue')
ax.plot(x,y_dist+offset*4,label=r'$\tau$ = 0.02 au',lw=2,c='dodgerblue')
ax.fill_between(x,y_dist+offset*4,offset*4,color='dodgerblue',alpha=0.2)
ax.axhline(offset*4,ls='dashed',lw=0.5,c='dodgerblue')
t = ax.text(-75,0.12+offset*4,r'$\tau$ = 0.02 au',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='dodgerblue'))
ax.text(-48,offset*4 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*4 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for 0.0100 timestep
y_dist, mean, error = gauss_tau(x,DLA_binding_energy,code_list_tau_DLA,0.0100)
ax.errorbar(mean,offset*3 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='turquoise')
ax.plot(x,y_dist+offset*3,label=r'$\tau$ = 0.01 au',lw=2,c='turquoise')
ax.fill_between(x,y_dist+offset*3,offset*3,color='turquoise',alpha=0.2)
ax.axhline(offset*3,ls='dashed',lw=0.5,c='turquoise')
t = ax.text(-75,0.12+offset*3,r'$\tau$ = 0.01 au',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='turquoise'))
ax.text(-48,offset*3 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*3 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for 0.0050 timestep
y_dist, mean, error = gauss_tau(x,DLA_binding_energy,code_list_tau_DLA,0.0050)
ax.errorbar(mean,offset*2 - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='orange')
ax.plot(x,y_dist+offset*2,label=r'$\tau$ = 0.005 au',lw=2,c='orange')
ax.fill_between(x,y_dist+offset*2,offset*2,color='orange',alpha=0.2)
ax.axhline(offset*2,ls='dashed',lw=0.5,c='orange')
t = ax.text(-75,0.12+offset*2,r'$\tau$ = 0.005 au',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='orange'))
ax.text(-48,offset*2 + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset*2 + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for 0.0025 timestep
y_dist, mean, error = gauss_tau(x,DLA_binding_energy,code_list_tau_DLA,0.0025)
ax.errorbar(mean,offset - 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='firebrick')
ax.plot(x,y_dist+offset,label=r'$\tau$ = 0.0025 au', lw=2,c='firebrick')
ax.fill_between(x,y_dist+offset,offset,color='firebrick',alpha=0.2)
ax.axhline(offset,ls='dashed',lw=0.5,c='firebrick')
t = ax.text(-75,0.12+offset,r'$\tau$ = 0.0025 au',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='firebrick'))
ax.text(-48,offset + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48,offset + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


# Plot data for timestep -> 0
y_dist, mean, error = gauss(x,final_binding_energy,method_codes_dict['DLA'],'DLA',dist_info=True)
ax.errorbar(mean,- 0.1,xerr=error,fmt='o',capsize=4,ms=8,elinewidth=2,markeredgewidth=2,c='red')
ax.plot(x,y_dist,label=r'$\tau$ $\to$ 0 au', lw=2,c='red')
ax.fill_between(x,y_dist,0,color='red',alpha=0.2)
ax.axhline(0,ls='dashed',lw=0.5,c='red')
t = ax.text(-75,0.12,r'$\tau$ $\to$ 0 au',fontsize=11)
t.set_bbox(dict(facecolor='none', alpha=0.5, edgecolor='red'))
ax.text(-48, + 0.2,r'$\mu_\tau \, {=}\,$' + f'{int(round(mean)):>-3} meV',fontsize=8)
ax.text(-48, + 0.1,r'$\sigma_\tau \, {=}\,$'+ f'{int(round(error)):>-3} meV',fontsize=8)


ax.spines[['right', 'top','left']].set_visible(False)

plt.yticks([])
plt.xlim([-85,-10])
plt.xlabel('Binding energy [meV]')

plt.tight_layout()
plt.savefig('Figures/Fig_03.png')

## Figure S1 - Time-step dependence of binding energy of individual codes

In [37]:
fig, axs = plt.subplots(2,4, figsize=(6.69,4.5), sharey=True, sharex=True,dpi=600,constrained_layout=True)

# Plot the top row binding energies
counter = 0
for counter, method in enumerate(['LA','TM','DLA','DLTM']):
    for code in ['CASINO', 'CMQMC','QMCPACK', 'TurboRVB_DMC','TurboRVB_LRDMC']:
        if code in method_codes_dict[method]:
            fitting_data = dmc_code_data[code]['binding_energy']['fit'][method]
            axs[0,counter].errorbar(fitting_data['x'], fitting_data['E'],yerr = fitting_data['E_err'], fmt='o', markersize=3,elinewidth=1, label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])
            axs[0,counter].plot( fitting_data['x_fit'], fitting_data['E_fit'], '--', color=dmc_code_info[code]['color'], alpha = 0.6 )
            axs[0,counter].fill_between(fitting_data['x_fit'],fitting_data['E_fit_lower_lim'],fitting_data['E_fit_upper_lim'],color=dmc_code_info[code]['color'],alpha=0.2)
    axs[0,counter].axhline(y=final_ccsdtq_ref,color='black',ls='--',label='CCSD(T)')


# Plot the top row binding energies
for counter, method in enumerate(['LA','TM','DLA']):
    for code in ['Amolqc','CHAMP_Holland','CHAMP_Cornell_Paris','PyQMC','QMC=Chem','QMeCha','QWalk']:
        if code in method_codes_dict[method]:
            fitting_data = dmc_code_data[code]['binding_energy']['fit'][method]
            axs[1,counter].errorbar(fitting_data['x'], fitting_data['E'],yerr = fitting_data['E_err'], fmt='o', markersize=3,elinewidth=1, label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])
            axs[1,counter].plot( fitting_data['x_fit'], fitting_data['E_fit'], '--', color=dmc_code_info[code]['color'], alpha = 0.6 )
            axs[1,counter].fill_between(fitting_data['x_fit'],fitting_data['E_fit_lower_lim'],fitting_data['E_fit_upper_lim'],color=dmc_code_info[code]['color'],alpha=0.2)
    axs[1,counter].axhline(y=final_ccsdtq_ref,color='black',ls='--',label='CCSD(T)')

# Create the legend
for code in dmc_code_info:
    axs[1,3].bar([1],[5], label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])

axs[1,3].axis('off')
axs[1,3].legend(fancybox=False, fontsize=7)

axs[0,0].set_title('LA')
axs[0,1].set_title('TM')
axs[0,2].set_title('DLA')
axs[0,3].set_title('DTM')

axs[0,0].set_xlim([0.0,0.09])
axs[0,0].set_ylim([-70,-20])
axs[0,0].set_ylabel('Binding energy [meV]')
axs[1,0].set_ylabel('Binding energy [meV]')

axs[1,0].set_xticks([0.00,0.03,0.06,0.09])
axs[1,0].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,1].set_xticks([0.00,0.03,0.06,0.09])
axs[1,1].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,2].set_xticks([0.00,0.03,0.06,0.09])
axs[1,2].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[1,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,1].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[1,2].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )

plt.savefig('Figures/Fig_S1.png')

## Figure S2 - Time-step dependence of total energies of individual codes

In [55]:
fig, axs = plt.subplots(3,4, figsize=(6.69,7),  sharey=True, sharex=True,dpi=600,constrained_layout=True)

final_total_energy = {code: {method: {system: dmc_code_data[code]['total_energy']['final'][f'{system}_{method}'] for system in ['mw', 'm', 'w']} for method in dmc_code_info[code]['methods_and_folders'].keys()}  for code in dmc_code_info.keys()}


for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
    counter = 0
    for indexj, method in enumerate(['LA','TM','DLA','DLTM']):
        mean_ene = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=tot_ene_sys_label)[0]
        mean_ene_std = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=tot_ene_sys_label)[1]
        for code in method_codes_dict[method]:
            fitting_data = dmc_code_data[code]['total_energy']['fit'][f'{tot_ene_sys_label}_{method}']

            axs[indexsys,indexj].errorbar(fitting_data['x'], fitting_data['E'] - mean_ene,yerr = fitting_data['E_err'], fmt='o', markersize=3,elinewidth=1, color=dmc_code_info[code]['color'])
            axs[indexsys,indexj].plot( fitting_data['x_fit'], fitting_data['E_fit'] - mean_ene, '--', linewidth=1, color=dmc_code_info[code]['color'], alpha = 0.6 )
            axs[indexsys,indexj].fill_between(fitting_data['x_fit'],fitting_data['E_fit_lower_lim'] - mean_ene,fitting_data['E_fit_upper_lim'] - mean_ene,color=dmc_code_info[code]['color'],alpha=0.2)

        axs[indexsys,indexj].axhline(0.0,c='grey',ls='dashed',lw=1, label=f'Avg={mean_ene:6.4f}(' f'{int(round(mean_ene_std*10000))})')
        axs[indexsys,indexj].legend(loc='upper right',fontsize=8)

        axs[indexsys,indexj].set_ylim([- 0.002, + 0.002])


for code in dmc_code_info:
    axs[0,3].bar([1],[5],label=dmc_code_info[code]['label'], color=dmc_code_info[code]['color'])


axs[0,0].set_title('LA')
axs[0,1].set_title('TM')
axs[0,2].set_title('DLA')
axs[0,3].set_title('DTM')

axs[0,0].set_xlim([0.0,0.09])
axs[0,0].set_ylabel('Methane--Water')
axs[1,0].set_ylabel('Methane')
axs[2,0].set_ylabel('Water ')
fig.supylabel('Deviation from average total energy [Ha]')


axs[2,0].set_xticks([0.00,0.03,0.06,0.09])
axs[2,0].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,1].set_xticks([0.00,0.03,0.06,0.09])
axs[2,1].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,2].set_xticks([0.00,0.03,0.06,0.09])
axs[2,2].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,3].set_xticks([0.00,0.03,0.06,0.09])
axs[2,3].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')

axs[2,0].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[2,1].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[2,2].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
axs[2,3].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )

plt.savefig('Figures/Fig_S2.png')

## Figure S3 - Deivation of zero time-step total energy of methane-water dimer, methane, water

In [59]:
final_total_energy['TurboRVB_DMC']

{'TM': {'mw': [-25.3215115251526, 9.069972965747297e-05],
  'm': [-8.080953311664542, 4.952998151175336e-05],
  'w': [-17.239527689921548, 6.187934252110949e-05]}}

In [60]:
sys_to_nice_name_dict = {
    'mw': 'Methane-Water',
    'm': 'Methane',
    'w': 'Water'
}

# figure 1 Comparison among different codes 
fig , ax = plt.subplots(4,3,figsize=(6.67,8),dpi=600,constrained_layout=True,sharex=True, sharey=True)

final_total_energy = {code: {method: {system: dmc_code_data[code]['total_energy']['final'][f'{system}_{method}'] for system in ['mw', 'm', 'w']} for method in dmc_code_info[code]['methods_and_folders'].keys()}  for code in dmc_code_info.keys()}

for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
    counter = 0
    for method in ['LA','TM','DLA','DLTM']:
        mean_ene = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=tot_ene_sys_label)[0]
        mean_ene_std = mean_codes_tot_ene(final_total_energy,method_codes_dict[method],method,syslabel=tot_ene_sys_label)[1]
        for code in method_codes_dict[method]:
            ax[counter,indexsys].errorbar(code_to_num_dict[code],final_total_energy[code][method][tot_ene_sys_label][0] - mean_ene,final_total_energy[code][method][tot_ene_sys_label][1],fmt='o',capsize=5,ms=6,elinewidth=1.5,markeredgewidth=1,mec='black',c=dmc_code_info[code]['color'])


        ax[counter,indexsys].axhline(0.0,c='grey',ls='dashed',lw=2, label=f'Avg={mean_ene:6.4f}(' f'{int(round(mean_ene_std*10000))})')
        ax[counter,indexsys].fill_between(np.arange(-0.5,11.5,0.1),0.0 + mean_ene_std,0.0 - mean_ene_std,color='grey',alpha=.3)
        ax[counter,indexsys].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
        ax[counter,indexsys].set_xticklabels([])
        ax[counter,indexsys].set_xlim([-0.5,11.5])
        ax[counter,indexsys].set_ylim([- 0.002, + 0.002])
        ax[counter,indexsys].legend()
        ax[counter,indexsys].set_title(f'{method} - {sys_to_nice_name_dict[tot_ene_sys_label]}')
        counter += 1

fig.supylabel('Deviation from average total energy [Ha]')

ax[3,0].set_xticklabels([dmc_code_info[code]['label'] for code in dmc_code_info],rotation=90) 
ax[3,1].set_xticklabels([dmc_code_info[code]['label'] for code in dmc_code_info],rotation=90) 
ax[3,2].set_xticklabels([dmc_code_info[code]['label'] for code in dmc_code_info],rotation=90) 

plt.savefig('Figures/Fig_S3.png')

# Figure S4 to 14 - Total energy and binding energy dependence

In [12]:
method_colors = {
    'LA': color_dict['red'],
    'TM': color_dict['blue'],
    'DLA': color_dict['green'],
    'DLTM': color_dict['orange']
}

for indexc, code_name in enumerate(dmc_code_info.keys()):
    fig, axs = plt.subplots(2,3,figsize=(6.69,5),constrained_layout=True, sharey='row', gridspec_kw={'height_ratios': [1,1.5]}, dpi=600)
    gs = axs[1, 2].get_gridspec()
    # remove the underlying axes
    for ax in axs[0,:]:
        ax.remove()
    axbig = fig.add_subplot(gs[0,:])
    for method in ['LA','TM','DLA','DLTM']:
        if code_name in method_codes_dict[method]:
            j1 = f'Eb_{method}'
            axbig.errorbar(dataEb_fit[code_name][j1]['x'], dataEb_fit[code_name][j1]['E'],yerr = dataEb_fit[code_name][j1]['E_err'], fmt='o', markersize=3,elinewidth=1, color=method_colors[method])
            axbig.plot( dataEb_fit[code_name][j1]['x_fit'], dataEb_fit[code_name][j1]['E_fit'], '--', linewidth=1, color=method_colors[method], alpha = 0.6, label=f'{method} -- ' + r'$E^\textrm{bind}_{\tau  \to 0} = $ ' + f'{final_binding_energy[code_name][method][0]:3.1f}' + r'${\pm}$' + f'{final_binding_energy[code_name][method][1]:3.1f} meV')
            axbig.fill_between(dataEb_fit[code_name][j1]['x_fit'],dataEb_fit[code_name][j1]['E_fit_lower_lim'],dataEb_fit[code_name][j1]['E_fit_upper_lim'],color=method_colors[method],alpha=0.2)
            j = f'E_{method}'

            for indexsys, tot_ene_sys_label in enumerate(['mw','m','w']):
                final_sys_ene = final_total_ene[code_name][method][tot_ene_sys_label][0]
                axs[1,indexsys].errorbar(data_total_ene_fit[code_name][j][tot_ene_sys_label]['x'], data_total_ene_fit[code_name][j][tot_ene_sys_label]['E'] - final_sys_ene,yerr = data_total_ene_fit[code_name][j][tot_ene_sys_label]['E_err'], fmt='o', markersize=3,elinewidth=1, color=method_colors[method])
                axs[1,indexsys].plot( data_total_ene_fit[code_name][j][tot_ene_sys_label]['x_fit'], data_total_ene_fit[code_name][j][tot_ene_sys_label]['E_fit'] - final_sys_ene, '--', linewidth=1, color=method_colors[method], alpha = 0.6, label=r'$E^\textrm{tot}_{\tau  \to 0}=$ ' +f'{final_sys_ene:6.4f}' )
                axs[1,indexsys].fill_between(data_total_ene_fit[code_name][j][tot_ene_sys_label]['x_fit'],data_total_ene_fit[code_name][j][tot_ene_sys_label]['E_fit_lower_lim'] - final_sys_ene ,data_total_ene_fit[code_name][j][tot_ene_sys_label]['E_fit_upper_lim'] - final_sys_ene,color=method_colors[method],alpha=0.2)
                # axs[indexsys+1].axhline(0.0,c='grey',ls='dashed',lw=1)

    axbig.set_ylabel('Binding energy [meV]')
    axbig.legend(ncol=2, loc='upper left', fontsize=8)
    axbig.set_xlim([0.0,0.09])
    axbig.set_ylim([-70,0])
    axbig.set_title(f'{code_final_to_nice_list[code_name]}')


    for i123 in range(3):
        axs[1,i123].legend(fontsize=8, loc = 'lower left')
        axs[1,i123].set_xlim([0.0,0.09])
        axs[1,i123].set_ylim([-0.007, 0.005])

    axs[1,0].set_ylabel(r'Deviation from $E^\textrm{tot}_{\tau  \to 0}$ [Ha]')
    axs[1,0].set_title('Methane--Water')
    axs[1,1].set_title('Methane')
    axs[1,2].set_title('Water')

    if code_name == 'TurboRVB':
        axbig.set_xlabel( r'LRDMC $a^2$ [a.u.]' ) 
        axs[1,0].set_xlabel( r'LRDMC $a^2$ [a.u.]' )
        axs[1,1].set_xlabel( r'LRDMC $a^2$ [a.u.]' )
        axs[1,2].set_xlabel( r'LRDMC $a^2$ [a.u.]' )    
    else:
        axbig.set_xlabel( r'DMC $\tau$ [a.u.]' )
        axs[1,0].set_xlabel( r'DMC $\tau$ [a.u.]' )
        axs[1,1].set_xlabel( r'DMC $\tau$ [a.u.]' )
        axs[1,2].set_xlabel( r'DMC $\tau$ [a.u.]' )

            
    plt.savefig(f'Figures/Fig_S{5+indexc}_{code_name}.png')


In [14]:
# for indexc, code_name in enumerate(['Amolqc','CASINO','CHAMP_Cornell_Paris','CHAMP_Holland','CMQMC','PyQMC','QMC=Chem','QMCPACK_final','QMeCha','QWalk','TurboRVB','TurboRVB_DMCv2']):
#     available_methods = [x for x in ['LA','TM','DLA','DLTM'] if code_name in method_codes_dict[x]]
#     available_methods_str = ", ".join(available_methods)
#     nice_code_name = code_name.replace("_","-")
#     print(r"\subsection{" + nice_code_name + r"}")
#     premable = "\\" + """begin{figure*}[!h]
#     \includegraphics[width=6.66in]{"""+ f"Figures/Fig_S{5+indexc}_{code_name}.png" + """}
#     \caption{\label{fig:""" + f"Code_plot_{code_name}" + """} The time step dependence of the (a) methane-water binding energy (b) methane-water dimer total energy, (c) isolated methane molecule total energy, (d) isolated water molecule total energy in the""" + f" {nice_code_name} code across the {available_methods_str} algorithm(s)." + """}
# \end{figure*}
# """ + """
# \clearpage"""
#     print(premable)

## Figure S17 - Assessing self-consistency error in binding energy

In [15]:
# Getting data for LA0 and TM0

dataEb['QMCPACK_final'] = {'Eb_SCE_LA': dataEb['QMCPACK_complex']['Eb_SCE_LA'],
                               'Eb_SCE_TM': dataEb['QMCPACK']['Eb_SCE_TM'],
                               'Eb_SCE_DLA': dataEb['QMCPACK']['Eb_SCE_DLA']}


# dataEb['QMCPACK_final'] = dataEb['QMCPACK']
dataEb['CASINO'] = dataEb['CASINO_limdmc5']


method_to_subplt_dict = {
    'LA': (0,0),
    'TM': (0,1),
    'DLA': (1,0),
    'DLTM': (1,1)
}


# Plotting the difference between DLA and DLTM against LA0 and TM0 for the water molecule system

fig, axs = plt.subplots(2,2, figsize=(6.69,5), sharey=True, sharex=True, dpi=600, constrained_layout=True )

fig.supylabel('Size Consistency Error [meV]' )
fig.supxlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )

for i, method in enumerate(methods):
    # axs[method_to_subplt_dict[method]].set_xlim(0,taumax)
    axs[method_to_subplt_dict[method]].set_xlim(1e-3,0.1); axs[method_to_subplt_dict[method]].set_xscale('log')

    # axs[i].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
    for code in code_final_list:
        try:
            datac1 = dataEb[code][f'Eb_SCE_{method}']
            datac1['x'] = x_axis( datac1, code )
            datac = datac1[ datac1.x <= taumax ]
            x = datac.x
            y = datac.ene
            y_err = datac.err
            axs[method_to_subplt_dict[method]].errorbar( x, y, y_err, fmt='o:', 
                           label=code, color=colors[code] )
        except:
            pass
    axs[method_to_subplt_dict[method]].axhline(y=0.0,color='black',ls='--')
    # axs[method_to_subplt_dict[method]].set_xticks([0.00,0.03,0.06,0.09])
    # axs[method_to_subplt_dict[method]].set_xticklabels([0.00,0.03,0.06,0.09],rotation=60,ha='center')
    axs[method_to_subplt_dict[method]].set_title(method)
    axs[method_to_subplt_dict[method]].set_ylim([-60,20])



leg = {}
for i in range(2):
    for k in range(2):
        handles, labels = axs[i,k].get_legend_handles_labels()
        for j in range(len(labels)):
            leg[code_final_to_nice_list[labels[j]]] = handles[j]
import collections
leg = collections.OrderedDict(sorted(leg.items()))
axs[1,1].legend(labels=list(leg.keys()), handles=list(leg.values()), ncol=2, loc='lower left',fontsize=7.5)

fig.savefig('Figures/Fig_S17.png')

## Figure S18 - Validation of DLA and DLTM algorithms with no Jastrow

In [16]:
# Getting data for LA0 and TM0

data_LA0 = {}

# Setting up the dictionary for methods to compare (i.e. DLA against LA0 and DLTM against TM0)
method2={
    'DLA': 'LA0',
    'DLTM': 'TM0'
}

for code in codes_all:
    data_LA0[code] = {}

for code in codes_all:
    if code.startswith('TurboRVB') and not ('DMC' in code):
        names = ['a','ene','err']
    else:
        names = ['tau','ene','err']
    for method in ['DLA','DLTM','LA0','TM0']:
        if 'QMCPACK' in code and method == 'DLTM':
            continue
        for syst in systems:
            key = f'{syst}_{method}'
            try: 
                data_LA0[code][key] = pd.read_csv(
                    f'../{code}/energy_{key}.csv',
                    names=names, decimal='.',
                    header=None, index_col=names[0] )
            except:
                data_LA0[code][key] = None
                #print(f'No data for {code}, {method}, {syst}')

data_LA0['QMCPACK_final'] = data_LA0['QMCPACK']
data_LA0['CASINO'] = data_LA0['CASINO_limdmc5']


# Plotting the difference between DLA and DLTM against LA0 and TM0 for the water molecule system

fig, axs = plt.subplots(1,2, figsize=(6.69,4), sharey=True, sharex=True, dpi=600, constrained_layout=True )

for i, method in enumerate(['DLA','DLTM']):
    axs[i].axhline(y=0.0,color='black',ls='--',label='')
    axs[i].set_xlim(0,0.025)
    axs[i].set_xlabel( r'DMC $\tau$ or LRDMC $a^2$ [a.u.]' )
    for code in ['Amolqc','CASINO','CMQMC','QMC=Chem','QMCPACK','QMeCha','TurboRVB']:
        try:
            datac1 = data_LA0[code][f'w_{method}']
            datac0 = data_LA0[code][f'w_{method2[method]}']
            datac = datac1 - datac0
            datac.err = (( datac1.err**2 + datac0.err**2 )**0.5)
            #print(code,method,'\n',datac0,datac1,datac)
            x = x_axis( datac, code )
            y = datac.ene
            y_err = datac.err
            axs[i].errorbar( x, y, y_err, fmt='o:', 
                            label=code, color=colors[code] )
        except:
            pass

axs[0].set_title('Determinant locality approximation')
axs[1].set_title('Determinant locality approximation with T-move')
axs[0].set_xlim((0.,0.021))
axs[0].set_ylim((-0.01,0.01))
axs[0].set_ylabel(r'Energy diff. between DLA (DTM) and LA0 (TM0) for H$_2$O [Ha]' )

leg = {}
for i in range(2): 
    handles, labels = axs[i].get_legend_handles_labels()
    for j in range(len(labels)):
        leg[color_final_to_nice_list[labels[j]]] = handles[j]
import collections
leg = collections.OrderedDict(sorted(leg.items()))
axs[1].legend(labels=list(leg.keys()), handles=list(leg.values()), ncol=2, loc='upper left',fontsize=8)

fig.savefig(f'Figures/Fig_S18.png')

NameError: name 'color_final_to_nice_list' is not defined